<a href="https://colab.research.google.com/github/Yuivika05/shesafe-app/blob/main/SheSafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
# Create the local environment file
with open(".env", "w") as f:
    f.write("GEMINI_API_KEY=AIzaSyYourActualKeyHere\n")

print("✓ .env file created successfully.")

✓ .env file created successfully.


In [66]:
# =====================================================================
# CELL 1: DEPENDENCIES & ENVIRONMENT SETUP (WITH .ENV LOADER)
# =====================================================================
!pip -q install gradio google-genai pillow reportlab python-dotenv

import gradio as gr
import hashlib
import hmac
import re
import json
import os
import datetime
import time
import urllib.parse
from PIL import Image
from google import genai
from dotenv import load_dotenv

# Load key directly from .env into environment
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_API_KEY", "")

# Secondary fallback to Colab secret storage if .env is missing
if not GEMINI_KEY:
    try:
        from google.colab import userdata
        GEMINI_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass

# File paths and security configurations
USERS_FILE = "secure_users_vault.json"
CASES_FILE = "secure_cases_vault.json"
SECRET_SALT = b"SheSafe_Secure_Dynamic_Salt_2026_#!"
EMAIL_REGEX = r"^[\w\.-]+@[\w\.-]+\.\w{2,}$"

print("✓ Cell 1: Environment loaded from .env successfully.")


✓ Cell 1: Environment loaded from .env successfully.


In [67]:
# =====================================================================
# CELL 2: SECURITY, VAULT STORAGE, GEMINI LEGAL ENGINE & NOTARIZATION
# =====================================================================

def hash_credential(password: str) -> str:
    return hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), SECRET_SALT, 100000).hex()

def load_json(path):
    if os.path.exists(path):
        try:
            with open(path, "r") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

# Initialize default demo account
users_db = load_json(USERS_FILE)
if "user@shesafe.org" not in users_db:
    users_db["user@shesafe.org"] = hash_credential("SafePassword@123")
    save_json(USERS_FILE, users_db)

def secure_login(email: str, password: str):
    email_clean = (email or "").strip().lower()
    pwd_clean = (password or "").strip()
    if not email_clean or not pwd_clean:
        return gr.update(visible=True), gr.update(visible=False), "⚠️ Email and password required.", ""
    u_db = load_json(USERS_FILE)
    if email_clean not in u_db or not hmac.compare_digest(hash_credential(pwd_clean), u_db[email_clean]):
        return gr.update(visible=True), gr.update(visible=False), "❌ Invalid credentials.", ""
    return gr.update(visible=False), gr.update(visible=True), "", email_clean

def secure_register(email: str, password: str, confirm_password: str):
    email_clean = (email or "").strip().lower()
    p, c = (password or "").strip(), (confirm_password or "").strip()
    if not email_clean or not p or not c:
        return "⚠️ All fields required."
    if not re.match(EMAIL_REGEX, email_clean):
        return "⚠️ Invalid email format."
    if len(p) < 8 or p != c:
        return "⚠️ Password must be 8+ characters and match confirmation."
    u_db = load_json(USERS_FILE)
    if email_clean in u_db:
        return "⚠️ Email already registered."
    u_db[email_clean] = hash_credential(p)
    save_json(USERS_FILE, u_db)
    return "✓ Account created! Sign in now."

def analyze_with_gemini(category: str, perpetrator: str, text: str, image_path: str) -> str:
    if not GEMINI_KEY:
        return "⚠️ Gemini API Key not found in .env. Please configure your .env file."

    try:
        client = genai.Client(api_key=GEMINI_KEY)
        prompt = f"""
Evaluate this reported safety incident under Indian statutory provisions:
Category: {category}
Perpetrator: {perpetrator or 'Anonymous'}
Evidence Transcript:
{text}

Provide:
1. Threat Risk Level: [CRITICAL | HIGH | MEDIUM | LOW]
2. Applicable Indian Law: (Sections under IT Act, 2000 and Bharatiya Nyaya Sanhita, 2023).
3. Immediate 2-hour technical precautions.
4. Redress summary ready for Cyber Cell submission.
"""
        contents = [prompt]
        if image_path and os.path.exists(image_path):
            try:
                contents.append(Image.open(image_path))
            except Exception:
                pass

        for m_name in ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash"]:
            try:
                response = client.models.generate_content(model=m_name, contents=contents)
                if response and response.text:
                    return response.text
            except Exception:
                continue
        return "❌ Error: Could not generate response from Gemini."
    except Exception as e:
        return f"❌ Connection Error: {str(e)}"

def process_evidence_and_ai(user_email: str, category: str, perpetrator: str, incident_text: str, image_file):
    if not user_email:
        return "⚠️ Session expired. Please log in again.", ""
    if not incident_text.strip() and image_file is None:
        return "⚠️ Please provide incident text or upload evidence.", ""

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")
    case_id = "SHESAFE-" + hashlib.sha1(f"{user_email}{timestamp}".encode()).hexdigest()[:8].upper()

    hasher = hashlib.sha256()
    hasher.update((incident_text or "").strip().encode('utf-8'))
    hasher.update(category.encode('utf-8'))
    hasher.update((perpetrator or "").strip().encode('utf-8'))

    img_name = None
    if image_file is not None:
        try:
            with open(image_file, "rb") as f:
                hasher.update(f.read())
            img_name = os.path.basename(image_file)
        except Exception:
            pass

    sha256_hash = hasher.hexdigest()
    ai_report = analyze_with_gemini(category, perpetrator, incident_text, image_file)

    cases_db = load_json(CASES_FILE)
    if user_email not in cases_db:
        cases_db[user_email] = []

    cases_db[user_email].append({
        "case_id": case_id,
        "timestamp": timestamp,
        "category": category,
        "perpetrator": perpetrator.strip() if perpetrator else "Anonymous",
        "description": incident_text.strip(),
        "evidence_file": img_name,
        "sha256": sha256_hash,
        "ai_report": ai_report
    })
    save_json(CASES_FILE, cases_db)

    result_md = (
        f"### 🛡️ Case Preserved: `{case_id}`\n"
        f"- **Timestamp:** `{timestamp}`\n"
        f"- **SHA-256 Digest:** `{sha256_hash}`\n\n"
        f"---\n{ai_report}\n\n"
        f"> ⚖️ *Assistive assessment for formal legal filing.*"
    )
    return result_md, render_case_history(user_email)

def render_case_history(user_email: str):
    cases_db = load_json(CASES_FILE)
    records = cases_db.get(user_email, [])
    if not records:
        return "*No incidents logged in your vault yet.*"
    md = "### 📂 Logged Case History\n\n"
    for r in reversed(records):
        md += f"---\n**Case ID:** `{r['case_id']}` | **Date:** {r['timestamp']} | **Category:** {r['category']}\n**SHA-256:** `{r['sha256']}`\n\n"
    return md

# Cryptographic Manifest & Notarization (Section 65B Indian Evidence Act Compliance)
def notarize_vault_record(case_id: str, user_email: str):
    cases_db = load_json(CASES_FILE)
    records = cases_db.get(user_email, [])
    target_case = next((c for c in records if c['case_id'] == case_id), None)

    if not target_case:
        return "⚠️ Case ID not found in vault."

    ist_tz = datetime.timezone(datetime.timedelta(hours=5, minutes=30))
    notary_time = datetime.datetime.now(ist_tz).strftime("%Y-%m-%d %H:%M:%S IST")

    # Canonical string construction for legal admissibility
    canonical_payload = (
        f"CASE:{target_case['case_id']}|"
        f"TIME:{target_case['timestamp']}|"
        f"CAT:{target_case['category']}|"
        f"PERP:{target_case['perpetrator']}|"
        f"ORIG_DIGEST:{target_case['sha256']}|"
        f"NOTARIZED:{notary_time}"
    )

    notarization_token = hmac.new(
        SECRET_SALT,
        canonical_payload.encode('utf-8'),
        hashlib.sha256
    ).hexdigest()

    target_case["notarization_token"] = notarization_token
    target_case["notarized_at"] = notary_time
    save_json(CASES_FILE, cases_db)

    receipt_md = (
        f"### 🛡️ Section 65B Digital Certificate Issued\n"
        f"- **Case ID:** `{target_case['case_id']}`\n"
        f"- **Canonical Digest:** `{target_case['sha256']}`\n"
        f"- **HMAC Notary Seal:** `{notarization_token}`\n"
        f"- **Certified At:** `{notary_time}`\n\n"
        f"> *Cryptographically sealed against post-incident tampering under Indian Evidence Act Section 65B.*"
    )
    return receipt_md

print("✓ Cell 2 restored: Cryptographic vault, Gemini AI engine & Notarization loaded successfully.")

✓ Cell 2 restored: Cryptographic vault, Gemini AI engine & Notarization loaded successfully.


In [68]:
# =====================================================================
# CELL 3: SATELLITE GPS & ZERO-NETWORK SOS MODULE
# =====================================================================

GPS_ACCURACY_LOCK_JS = """
async (contacts, note, landmark, lat, lon, acc) => {
    return new Promise((resolve) => {
        if (!navigator.geolocation) {
            resolve([contacts, note, landmark, "0", "0", "0"]);
            return;
        }
        let bestPos = null;
        let watchId = null;
        const timer = setTimeout(() => {
            if (watchId !== null) navigator.geolocation.clearWatch(watchId);
            if (bestPos) {
                resolve([contacts, note, landmark, bestPos.coords.latitude.toFixed(6), bestPos.coords.longitude.toFixed(6), bestPos.coords.accuracy.toFixed(1)]);
            } else {
                resolve([contacts, note, landmark, "0", "0", "0"]);
            }
        }, 3000);

        watchId = navigator.geolocation.watchPosition(
            (pos) => {
                if (!bestPos || pos.coords.accuracy < bestPos.coords.accuracy) {
                    bestPos = pos;
                }
                if (pos.coords.accuracy <= 25) {
                    clearTimeout(timer);
                    navigator.geolocation.clearWatch(watchId);
                    resolve([contacts, note, landmark, pos.coords.latitude.toFixed(6), pos.coords.longitude.toFixed(6), pos.coords.accuracy.toFixed(1)]);
                }
            },
            (err) => {
                clearTimeout(timer);
                if (watchId !== null) navigator.geolocation.clearWatch(watchId);
                resolve([contacts, note, landmark, "0", "0", "0"]);
            },
            { enableHighAccuracy: true, maximumAge: 0, timeout: 3500 }
        );
    });
}
"""

def trigger_sos_alert(contacts_raw: str, distress_note: str, manual_landmark: str, live_lat: str, live_lon: str, live_acc: str):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S IST")
    sos_id = "SOS-" + hashlib.sha256(f"{contacts_raw}{ts}".encode()).hexdigest()[:8].upper()

    lat_clean = (live_lat or "").strip()
    lon_clean = (live_lon or "").strip()
    acc_clean = (live_acc or "").strip()

    if lat_clean and lon_clean and lat_clean not in ["0", ""] and lon_clean not in ["0", ""]:
        lat = round(float(lat_clean), 6)
        lon = round(float(lon_clean), 6)
        acc_str = f"±{float(acc_clean):.1f}m" if acc_clean and acc_clean != "0" else "High"
        gps_status = f"🟢 Satellite GPS Hardware Lock ({acc_str})"
        maps_link = f"https://www.google.com/maps?q={lat},{lon}"
        embed_map_url = f"https://maps.google.com/maps?q={lat},{lon}&z=16&output=embed"
    elif manual_landmark.strip():
        encoded_query = urllib.parse.quote(manual_landmark.strip())
        lat, lon = "N/A", "N/A"
        gps_status = f"📍 Specific Landmark: {manual_landmark.strip()}"
        maps_link = f"https://www.google.com/maps/search/?api=1&query={encoded_query}"
        embed_map_url = f"https://maps.google.com/maps?q={encoded_query}&z=16&output=embed"
    else:
        lat, lon = 28.4944, 77.5378
        gps_status = "🟡 Regional Estimate (NCR / Greater Noida)"
        maps_link = f"https://www.google.com/maps?q={lat},{lon}"
        embed_map_url = f"https://maps.google.com/maps?q={lat},{lon}&z=16&output=embed"

    contact_list = [c.strip() for c in contacts_raw.split(",") if c.strip()]
    recipients = ", ".join(contact_list) if contact_list else "+91-112 (Police), Trusted Contacts"
    primary_num = contact_list[0] if contact_list else "112"

    loc_str = f"Coords: {maps_link}" + (f" | Landmark: {manual_landmark.strip()}" if manual_landmark.strip() else "")
    sms_text = f"EMERGENCY SOS: I need help. {loc_str} [Token: {sos_id}]. Note: {distress_note if distress_note else 'Unsafe'}. Time: {ts}"
    encoded_sms = urllib.parse.quote(sms_text)

    sms_uri = f"sms:{primary_num}?body={encoded_sms}"
    whatsapp_url = f"https://api.whatsapp.com/send?text={encoded_sms}"

    return f"""
<div style="background:#fff1f2; border:2px solid #e63946; border-radius:14px; padding:18px;">
    <div style="display:flex; justify-content:space-between; align-items:center;">
        <span style="font-size:16px; font-weight:700; color:#e63946;">🚨 EMERGENCY SOS DISPATCHED</span>
        <span style="background:#e63946; color:white; font-size:11px; padding:3px 8px; border-radius:8px; font-weight:bold;">ACTIVE</span>
    </div>
    <div style="margin-top:12px; font-size:13px; line-height:1.6; color:#333;">
        <b>GPS Status:</b> {gps_status}<br>
        <b>Recipients:</b> {recipients}<br>
        <b>Token:</b> <code>{sos_id}</code> &nbsp;|&nbsp; <b>Time:</b> {ts}<br>
        <b>Coordinates:</b> Lat <code>{lat}</code>, Lon <code>{lon}</code>
    </div>
    <div style="margin-top:14px; display:flex; gap:8px; flex-wrap:wrap;">
        <a href="{sms_uri}" style="background:#d90429; color:white; text-decoration:none; padding:8px 16px; border-radius:20px; font-weight:700; font-size:12px;">💬 Send 2G SMS (Zero Data)</a>
        <a href="{whatsapp_url}" target="_blank" style="background:#25D366; color:white; text-decoration:none; padding:8px 16px; border-radius:20px; font-weight:600; font-size:12px;">📲 Send WhatsApp</a>
        <a href="tel:112" style="background:#111; color:white; text-decoration:none; padding:8px 16px; border-radius:20px; font-weight:600; font-size:12px;">📞 Dial 112 (Police)</a>
        <a href="tel:181" style="background:#bd6875; color:white; text-decoration:none; padding:8px 16px; border-radius:20px; font-weight:600; font-size:12px;">📞 Dial 181 (Helpline)</a>
        <a href="{maps_link}" target="_blank" style="background:#4285F4; color:white; text-decoration:none; padding:8px 16px; border-radius:20px; font-weight:600; font-size:12px;">📍 Open Map Pin</a>
    </div>
    <div style="margin-top:14px; border-radius:10px; overflow:hidden; border:1px solid #ffccd5;">
        <iframe width="100%" height="200" frameborder="0" scrolling="no" src="{embed_map_url}"></iframe>
    </div>
</div>
"""

print("✓ Cell 3: Satellite GPS & zero-network SOS module ready.")

# =====================================================================
# CELL 3 ADDITION: HAVERSINE GEOFENCE DRIFT CALCULATOR (MODULE 3)
# =====================================================================

def compute_geofence_deviation(curr_lat: float, curr_lon: float, target_lat: float, target_lon: float, max_radius_meters: float):
    # Earth radius in meters
    R = 6371000.0

    try:
        lat1, lon1 = math.radians(float(curr_lat)), math.radians(float(curr_lon))
        lat2, lon2 = math.radians(float(target_lat)), math.radians(float(target_lon))

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        distance_meters = R * c

        deviated = distance_meters > float(max_radius_meters)
        return round(distance_meters, 2), deviated
    except Exception:
        return 0.0, False

✓ Cell 3: Satellite GPS & zero-network SOS module ready.


In [69]:
# =====================================================================
# CELL 4: COVERT ESCAPE CALL ENGINE
# =====================================================================

def start_incoming_call(delay_val: int, persona: str, scenario: str):
    time.sleep(int(delay_val))
    name = persona.strip() if persona.strip() else "Mom"
    top_html = f"""
    <div class="phone-top">
        <div class="phone-incoming-tag">INCOMING CALL...</div>
        <div class="phone-caller-name">{name}</div>
        <div class="phone-caller-context">Mobile | {scenario}</div>
    </div>
    <div class="phone-avatar-disc">
        <svg viewBox="0 0 24 24"><path d="M12 12c2.21 0 4-1.79 4-4s-1.79-4-4-4-4 1.79-4 4 1.79 4 4 4zm0 2c-2.67 0-8 1.34-8 4v2h16v-2c0-2.66-5.33-4-8-4z"/></svg>
    </div>
    """
    return gr.update(visible=False), gr.update(visible=True), top_html, gr.update(visible=True), gr.update(visible=False), name

def answer_call(active_name: str):
    connected_html = f"""
    <div class="phone-top">
        <div class="phone-incoming-tag" style="color:#2ec4b6;">● CALL CONNECTED</div>
        <div class="phone-caller-name">{active_name}</div>
        <div class="phone-caller-context" style="color:#2ec4b6;">00:10 • Connected</div>
    </div>
    <div class="connected-banner">
        <div class="connected-script">
            "Hey, where are you right now? Can you please step outside? I'm waiting near the gate."
        </div>
    </div>
    """
    return connected_html, gr.update(visible=False), gr.update(visible=True)

def end_call():
    return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

print("✓ Cell 4: Covert call simulation engine ready.")


✓ Cell 4: Covert call simulation engine ready.


In [70]:
# =====================================================================
# CELL 5: ADVANCED EXTENSIONS (UPDATED WITH ACTIVE JAVASCRIPT TIMER & ALARM)
# =====================================================================

# JavaScript countdown that runs directly in the user's browser:
TIMER_START_JS = f"""
(mins) => {{
    const duration = parseInt(mins) || 1;
    let secondsLeft = duration * 60;

    // Clear any previous running countdown
    if (window._safetyWalkInterval) {{
        clearInterval(window._safetyWalkInterval);
        window._safetyWalkInterval = null;
    }}

    // Target container elements
    const container = document.getElementById('timer_live_container');
    if (!container) return;

    // Render active countdown frame
    container.innerHTML = `
        <div style="text-align:center; padding:18px; background:#fff0f3; border:2px solid #ff4d6d; border-radius:14px; margin-top:12px;">
            <h3 style="color:#c9184a; margin:0 0 6px;">⏱️ 'Walk With Me' Countdown Armed</h3>
            <p style="margin:0 0 10px; font-size:13px; color:#555;">Timer active for <b>${{duration}} minute(s)</b></p>
            <div id="walk_clock_digits" style="font-size:42px; font-weight:800; color:#800f2f; letter-spacing:2px; font-family:monospace;">
                ${{String(Math.floor(secondsLeft / 60)).padStart(2, '0')}}:${{String(secondsLeft % 60).padStart(2, '0')}}
            </div>
            <div id="walk_audio_zone"></div>
        </div>
    `;

    // Start live 1-second interval loop
    window._safetyWalkInterval = setInterval(() => {{
        secondsLeft--;
        const clockEl = document.getElementById('walk_clock_digits');

        if (secondsLeft <= 0) {{
            clearInterval(window._safetyWalkInterval);
            window._safetyWalkInterval = null;

            if (clockEl) {{
                clockEl.innerText = "00:00 - EMERGENCY BLAST";
                clockEl.style.color = "#e63946";
            }}

            // Blast the offline Base64 Siren automatically
            const audioZone = document.getElementById('walk_audio_zone');
            if (audioZone) {{
                audioZone.innerHTML = `
                    <audio autoplay loop src="data:audio/wav;base64,{SIREN_B64}"></audio>
                    <div style="color:white; font-weight:800; padding:10px; background:#d90429; border-radius:8px; margin-top:12px; font-size:16px;">
                        🚨 TIME EXPIRED! EMERGENCY AUDITORY ALARM TRIGGERED 🚨
                    </div>
                `;
            }}
        }} else {{
            if (clockEl) {{
                const m = Math.floor(secondsLeft / 60);
                const s = secondsLeft % 60;
                clockEl.innerText = `${{String(m).padStart(2, '0')}}:${{String(s).padStart(2, '0')}}`;
            }}
        }}
    }}, 1000);
}}
"""

TIMER_CANCEL_JS = """
() => {
    if (window._safetyWalkInterval) {
        clearInterval(window._safetyWalkInterval);
        window._safetyWalkInterval = null;
    }
    const container = document.getElementById('timer_live_container');
    if (container) {
        container.innerHTML = `
            <div style="text-align:center; padding:14px; background:#f0fff4; border:1px solid #9ae6b4; border-radius:10px; color:#22543d; margin-top:12px; font-weight:600;">
                ✓ Safe Check-In Confirmed. Safety Timer Disarmed.
            </div>
        `;
    }
}
"""

# =====================================================================
# CELL 5 ADDITIONS: ACOUSTIC SENTINEL & LIVE ROUTE WATCHDOG (JS)
# =====================================================================

# 1. Passive Distress Listener JS (Module 1)
START_DISTRESS_LISTENER_JS = """
() => {
    try {
        if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
            alert('Microphone access is not supported in this browser environment.');
            return;
        }

        navigator.mediaDevices.getUserMedia({ audio: true }).then(stream => {
            const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            const source = audioCtx.createMediaStreamSource(stream);
            const analyser = audioCtx.createAnalyser();
            analyser.fftSize = 512;
            source.connect(analyser);

            window._audioStream = stream;
            window._distressCtx = audioCtx;
            const dataArray = new Uint8Array(analyser.frequencyBinCount);
            const statusBox = document.getElementById('distress_meter_status');

            let spikeCount = 0;
            window._distressMonitor = setInterval(() => {
                analyser.getByteFrequencyData(dataArray);
                let sum = 0;
                for (let i = 0; i < dataArray.length; i++) {
                    sum += dataArray[i];
                }
                let avg = sum / dataArray.length;

                // Threshold check: aggressive sound spike (> 75/255 scale)
                if (avg > 75) {
                    spikeCount++;
                    if (spikeCount >= 3) { // Sustained scream/distress spike
                        clearInterval(window._distressMonitor);
                        if (statusBox) {
                            statusBox.innerHTML = '<span style="color:#e63946; font-weight:800;">🚨 DISTRESS SPIKE DETECTED! ESCALATING ALARM...</span>';
                        }
                        // Automatically click the main SOS button
                        const sosBtn = document.querySelector('.danger-sos-btn');
                        if (sosBtn) sosBtn.click();
                    }
                } else {
                    spikeCount = Math.max(0, spikeCount - 1);
                    if (statusBox) {
                        statusBox.innerHTML = `<span style="color:#2ec4b6;">● Listening passively... Ambient level: ${Math.round(avg)}</span>`;
                    }
                }
            }, 150);
        }).catch(err => {
            const statusBox = document.getElementById('distress_meter_status');
            if (statusBox) statusBox.innerText = 'Mic Permission Denied: ' + err.message;
        });
    } catch(err) {
        console.error(err);
    }
}
"""

STOP_DISTRESS_LISTENER_JS = """
() => {
    if (window._distressMonitor) {
        clearInterval(window._distressMonitor);
        window._distressMonitor = null;
    }
    if (window._audioStream) {
        window._audioStream.getTracks().forEach(track => track.stop());
        window._audioStream = null;
    }
    if (window._distressCtx) {
        window._distressCtx.close();
        window._distressCtx = null;
    }
    const statusBox = document.getElementById('distress_meter_status');
    if (statusBox) statusBox.innerHTML = '<span style="color:#718096;">Sentinel Standby (Mic Off)</span>';
}
"""

# 2. Live Route Monitor JS (Module 3)
START_ROUTE_WATCHDOG_JS = """
(contact, destination, maxDrift) => {
    if (!navigator.geolocation) {
        alert('Geolocation is not supported on this device.');
        return;
    }

    const feedback = document.getElementById('route_watchdog_feedback');
    const dispatchBox = document.getElementById('route_dispatch_container');
    const phoneClean = (contact || '').replace(/[^0-9+]/g, '');

    if (feedback) {
        feedback.innerHTML = '<span style="color:#2ec4b6; font-weight:600;">● Route Watchdog Active: Polling GPS trajectory...</span>';
    }

    if (window._routeWatchId) navigator.geolocation.clearWatch(window._routeWatchId);

    window._routeWatchId = navigator.geolocation.watchPosition(
        (pos) => {
            const lat = pos.coords.latitude.toFixed(5);
            const lon = pos.coords.longitude.toFixed(5);
            const acc = Math.round(pos.coords.accuracy);

            const mapsLink = `https://maps.google.com/?q=${lat},${lon}`;
            const alertText = encodeURIComponent(`ROUTE DEVIATION ALERT! Heading towards: ${destination || 'Destination'}. Current Location: ${mapsLink} (±${acc}m)`);
            const smsUri = `sms:${phoneClean}?body=${alertText}`;
            const waUri = `https://wa.me/${phoneClean.replace('+', '')}?text=${alertText}`;

            if (feedback) {
                feedback.innerHTML = `<span>Active Track: Lat <b>${lat}</b>, Lon <b>${lon}</b> (Accuracy: ±${acc}m)</span>`;
            }

            if (dispatchBox) {
                dispatchBox.innerHTML = `
                    <div style="background:#fff0f3; border:2px solid #e63946; border-radius:12px; padding:15px; margin-top:14px;">
                        <div style="font-weight:700; color:#c9184a; margin-bottom:8px;">🚨 Direct Dispatch Ready for ${phoneClean || 'Emergency Contacts'}</div>
                        <p style="font-size:12px; color:#666; margin:0 0 10px;">Tap below to instantly forward your live coordinates if the route diverts:</p>
                        <div style="display:flex; gap:10px;">
                            <a href="${smsUri}" target="_blank" style="flex:1; text-align:center; background:#1d3557; color:white; padding:10px; border-radius:8px; text-decoration:none; font-weight:600; font-size:13px;">
                                📩 Open Phone SMS Inbox
                            </a>
                            <a href="${waUri}" target="_blank" style="flex:1; text-align:center; background:#25d366; color:white; padding:10px; border-radius:8px; text-decoration:none; font-weight:600; font-size:13px;">
                                💬 Open WhatsApp
                            </a>
                        </div>
                    </div>
                `;
            }
        },
        (err) => {
            if (feedback) feedback.innerText = 'GPS Error: ' + err.message;
        },
        { enableHighAccuracy: true, maximumAge: 5000, timeout: 10000 }
    );
}
"""

STOP_ROUTE_WATCHDOG_JS = """
() => {
    if (window._routeWatchId) {
        navigator.geolocation.clearWatch(window._routeWatchId);
        window._routeWatchId = null;
    }
    const feedback = document.getElementById('route_watchdog_feedback');
    if (feedback) feedback.innerHTML = '<span style="color:#718096;">Route Watchdog Idle</span>';
}
"""

In [71]:
# =====================================================================
# CELL 6: APPLICATION UI & LAUNCHER
# =====================================================================

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Playfair+Display:wght@500;600&display=swap');
* { box-sizing: border-box; }
body { margin: 0; font-family: 'DM Sans', sans-serif; background: #ffffff; color: #252525; }
.gradio-container { max-width: 1400px !important; margin: auto !important; padding: 0 !important; }

.navbar { height: 75px; display: flex; align-items: center; justify-content: space-between; padding: 0 40px; border-bottom: 1px solid #f1eeee; background: white; }
.logo { display: flex; align-items: center; gap: 8px; }
.logo-icon { color: #d98d98; font-size: 28px; }
.logo-text { font-family: 'Playfair Display', serif; font-size: 28px; color: #292929; }
.logo-text span { color: #d98d98; }
.nav-badge { background: #eef9f6; color: #1a7f64; border: 1px solid #b6eadb; padding: 4px 10px; border-radius: 12px; font-size: 12px; font-weight: 600; }

.auth-stage { min-height: 480px; display: flex; align-items: center; justify-content: space-between; padding: 30px 60px; gap: 40px; }
.auth-left { width: 48%; }
.hero-title { font-family: 'Playfair Display', serif; font-size: 48px; line-height: 1.1; font-weight: 500; color: #252525; }
.hero-title span { color: #cc7b86; }
.hero-description { font-size: 15px; color: #666; line-height: 1.6; margin: 18px 0; }
.auth-card { width: 48%; background: #ffffff; border: 1px solid #eee; border-radius: 20px; padding: 30px 35px; box-shadow: 0 8px 25px rgba(0,0,0,0.03); }

.primary-btn { background: #d98793 !important; color: white !important; border: none !important; border-radius: 25px !important; padding: 12px 24px !important; font-weight: 600 !important; cursor: pointer !important; }
.danger-sos-btn { background: #e63946 !important; color: white !important; border: none !important; border-radius: 30px !important; padding: 16px 28px !important; font-weight: 700 !important; font-size: 15px !important; cursor: pointer !important; }

.vault-panel { background: #ffffff; border: 1px solid #f1eeee; border-radius: 18px; padding: 25px; margin: 15px 40px 30px; box-shadow: 0 6px 20px rgba(0,0,0,0.03); }

.phone-viewport-container { display: flex !important; flex-direction: column !important; align-items: center !important; }
.phone-screen { width: 320px !important; height: 520px !important; background: #12131a !important; border-radius: 40px !important; padding: 35px 20px 30px !important; box-shadow: 0 20px 50px rgba(0,0,0,0.4) !important; border: 3px solid #23242e !important; display: flex !important; flex-direction: column !important; justify-content: space-between !important; align-items: center !important; box-sizing: border-box !important; }
.phone-top { text-align: center; width: 100%; }
.phone-incoming-tag { font-size: 11px; letter-spacing: 2px; color: #8c8ea3; text-transform: uppercase; font-weight: 600; }
.phone-caller-name { font-size: 28px; font-weight: 700; color: #ffffff; margin-top: 8px; }
.phone-caller-context { font-size: 13px; color: #9294ab; margin-top: 4px; }
.phone-avatar-disc { width: 85px; height: 85px; border-radius: 50%; background: #20222f; display: flex; align-items: center; justify-content: center; margin: 15px auto; }
.phone-avatar-disc svg { width: 42px; height: 42px; fill: #7c74a0; }
.phone-actions-row { display: flex !important; flex-direction: row !important; justify-content: space-around !important; align-items: center !important; width: 100% !important; }
.btn-column { display: flex !important; flex-direction: column !important; align-items: center !important; gap: 6px !important; }
.btn-circle { width: 62px !important; height: 62px !important; border-radius: 50% !important; border: none !important; display: flex !important; align-items: center !important; justify-content: center !important; cursor: pointer !important; font-size: 24px !important; }
.btn-decline-red { background: #e63946 !important; color: white !important; }
.btn-accept-green { background: #2ec4b6 !important; color: white !important; }
.connected-banner { background: #192b27; border: 1px solid #2ec4b6; border-radius: 14px; padding: 14px; width: 100%; text-align: center; }
.connected-script { font-size: 13px; color: #e5f9f6; font-style: italic; }
"""

with gr.Blocks(title="SheSafe Platform") as app:
    current_user_state = gr.State("")
    current_caller_state = gr.State("Mom")

    # Stealth Disguise Panel (Calculator Screen)
    with gr.Column(visible=False) as disguise_panel:
        gr.Markdown("### Standard Calculator")
        calc_display = gr.Textbox(label="Display", interactive=True)
        gr.Markdown("*Enter an equation and press Calculate. (Type `9999` to return to SheSafe)*")
        calc_btn = gr.Button("Calculate", elem_classes=["primary-btn"])

    # Core SheSafe Platform Container
    with gr.Column(visible=True) as main_she_safe_container:
        gr.HTML("""
        <div class="navbar">
            <div class="logo"><div class="logo-icon">♢</div><div class="logo-text">She<span>Safe</span></div></div>
            <span class="nav-badge">● 2G / GSM Offline Ready</span>
        </div>
        """)

        with gr.Column(visible=True) as auth_panel:
            with gr.Row(elem_classes=["auth-stage"]):
                gr.HTML("""
                <div class="auth-left">
                    <h1 class="hero-title">Your Safe<br><span>Sanctuary.</span><br>Secured.</h1>
                    <p class="hero-description">Comprehensive women's safety portal with cryptographic evidence preservation, statutory assessment, and offline 2G SMS emergency fallback.</p>
                </div>
                """)
                with gr.Column(elem_classes=["auth-card"]):
                    with gr.Tabs():
                        with gr.TabItem("Sign In"):
                            login_email = gr.Textbox(label="Email Address", value="user@shesafe.org")
                            login_password = gr.Textbox(label="Password", type="password", value="SafePassword@123")
                            login_submit = gr.Button("Access Safe Vault →", elem_classes=["primary-btn"])
                            auth_feedback = gr.Markdown("")
                        with gr.TabItem("Register"):
                            reg_email = gr.Textbox(label="Email Address")
                            reg_password = gr.Textbox(label="Create Password", type="password")
                            reg_confirm = gr.Textbox(label="Confirm Password", type="password")
                            reg_submit = gr.Button("Create Secure Vault →", elem_classes=["primary-btn"])
                            reg_feedback = gr.Markdown("")

        with gr.Column(visible=False) as app_panel:
            with gr.Row(elem_classes=["vault-panel"]):
                with gr.Column(scale=3):
                    gr.Markdown("## 🌸 SheSafe Safety Dashboard")
                with gr.Column(scale=1):
                    stealth_trigger = gr.Button("🎭 Camouflage (Calculator)", elem_classes=["primary-btn"])
                with gr.Column(scale=1):
                    logout_btn = gr.Button("Sign Out", elem_classes=["primary-btn"])

            with gr.Row(elem_classes=["vault-panel"]):
                with gr.Tabs():
                    # TAB 1: FORENSIC AI & LEGAL VAULT
                    with gr.TabItem("⚖️ Forensic AI & Legal Vault"):
                        with gr.Row():
                            with gr.Column(scale=1):
                                cat_input = gr.Dropdown(label="Category", choices=["Online Harassment", "Deepfakes / Image Abuse", "Impersonation", "Cyber Blackmail"], value="Online Harassment")
                                perp_input = gr.Textbox(label="Perpetrator Identifier (Optional)")
                                narrative_input = gr.Textbox(label="Incident Transcript", lines=4)
                                file_input = gr.File(label="Upload Screenshot", file_types=["image"])
                                submit_btn = gr.Button("Run AI Legal Assessment ⚖️", elem_classes=["primary-btn"])
                                with gr.Row():
                                    gen_pdf_btn = gr.Button("📄 Police PDF", elem_classes=["primary-btn"])
                                    notarize_btn = gr.Button("🔏 Sec 65B Seal", elem_classes=["primary-btn"])
                                pdf_file_out = gr.File(label="Download Complaint PDF")
                            with gr.Column(scale=1):
                                vault_status_output = gr.Markdown("*Submit incident to view assessment.*")
                                case_history_output = gr.Markdown("")

                    # TAB 2: EMERGENCY SOS & GEO-TRACE
                    with gr.TabItem("🚨 Emergency SOS & Geo-Trace"):
                        with gr.Row():
                            with gr.Column(scale=1):
                                sos_contacts = gr.Textbox(label="Emergency Contacts", value="+91-9876543210")
                                sos_note = gr.Textbox(label="Distress Note", placeholder="Feeling unsafe...")
                                landmark_box = gr.Textbox(label="Current Landmark / Building (Optional)", placeholder="e.g. Campus Gate 2, Hostel Block A...")
                                live_lat_box = gr.Textbox(visible=False, value="0")
                                live_lon_box = gr.Textbox(visible=False, value="0")
                                live_acc_box = gr.Textbox(visible=False, value="0")
                                sos_btn = gr.Button("🚨 TRIGGER EMERGENCY SOS", elem_classes=["danger-sos-btn"])
                            with gr.Column(scale=1):
                                sos_display = gr.HTML("<p style='color:#777;'>Armed. Tap SOS to lock location.</p>")

                    # TAB 3: COVERT FAKE CALL
                    with gr.TabItem("📱 Covert Fake Call"):
                        with gr.Row():
                            with gr.Column(scale=1):
                                fake_name = gr.Dropdown(label="Caller", choices=["Mom", "Dad", "Hostel Warden", "Cab Driver"], value="Mom")
                                fake_type = gr.Dropdown(label="Scenario", choices=["Urgent Emergency", "Cab Arrived", "Security Check"], value="Urgent Emergency")
                                call_delay = gr.Radio(label="Delay", choices=[("0s", 0), ("5s", 5), ("15s", 15)], value=0)
                                call_btn = gr.Button("📞 Trigger Escape Call", elem_classes=["primary-btn"])
                            with gr.Column(scale=1, elem_classes=["phone-viewport-container"]):
                                call_standby_box = gr.HTML("<p style='color:#888; padding:80px 0;'>No active incoming call.</p>", visible=True)
                                with gr.Column(visible=False, elem_classes=["phone-screen"]) as phone_viewport:
                                    phone_info_html = gr.HTML("")
                                    with gr.Row(visible=False, elem_classes=["phone-actions-row"]) as incoming_actions_row:
                                        with gr.Column(elem_classes=["btn-column"]):
                                            decline_call_btn = gr.Button("✕", elem_classes=["btn-circle", "btn-decline-red"])
                                        with gr.Column(elem_classes=["btn-column"]):
                                            accept_call_btn = gr.Button("📞", elem_classes=["btn-circle", "btn-accept-green"])
                                    with gr.Row(visible=False, elem_classes=["phone-actions-row"]) as hangup_actions_row:
                                        hangup_call_btn = gr.Button("✕", elem_classes=["btn-circle", "btn-decline-red"])

                    # TAB 4: HIGH-DECIBEL DETERRENT SIREN
                    with gr.TabItem("📢 Deterrent Siren"):
                        with gr.Column(elem_classes=["vault-panel"]):
                            gr.HTML("""
                            <div style="text-align:center; padding: 10px 0 16px;">
                                <h2 style="color:#e63946; margin:0;">📢 High-Decibel Deterrent Siren</h2>
                                <p style="color:#666; font-size:14px; margin-top:4px;">Direct media audio pipeline. Works 100% offline.</p>
                            </div>
                            """)
                            with gr.Row():
                                siren_start_btn = gr.Button("▶ ACTIVATE SIREN", elem_classes=["danger-sos-btn"])
                                siren_stop_btn = gr.Button("⏹ SILENCE", elem_classes=["primary-btn"])
                            siren_audio_sink = gr.HTML("<div style='text-align:center; color:#718096; padding:8px;'>Standby</div>")

                    # TAB 5: SAFETY COUNTDOWN TIMER
                    with gr.TabItem("⏱️ Safety Timer"):
                        with gr.Column(elem_classes=["vault-panel"]):
                            gr.Markdown("### 🚶‍♀️ 'Walk With Me' Live Countdown Timer\nSet a safety timer when walking alone. A real-time countdown begins on your screen.")
                            timer_mins = gr.Radio(label="Select Duration", choices=[("1 Min (Test)", 1), ("5 Mins", 5), ("15 Mins", 15)], value=1)
                            with gr.Row():
                                timer_start_btn = gr.Button("⏱️ Arm Safety Timer", elem_classes=["primary-btn"])
                                timer_cancel_btn = gr.Button("✓ I Am Safe (Cancel Timer)", elem_classes=["primary-btn"])
                            gr.HTML("<div id='timer_live_container' style='color:#718096; padding:12px 0;'>Timer inactive.</div>")

                    # TAB 6: ACOUSTIC DISTRESS SENTINEL
                    with gr.TabItem("🎙️ Distress Sentinel"):
                        with gr.Column(elem_classes=["vault-panel"]):
                            gr.Markdown("### 🎙️ Passive Acoustic Screaming & Distress Monitor\nRuns hands-free in browser RAM. Automatically trips SOS dispatch if screaming or physical struggle audio spikes occur.")
                            with gr.Row():
                                start_sentinel_btn = gr.Button("🛡️ Arm Acoustic Sentinel", elem_classes=["primary-btn"])
                                stop_sentinel_btn = gr.Button("⏹ Disarm Sentinel", elem_classes=["primary-btn"])
                            gr.HTML("<div id='distress_meter_status' style='padding:12px; font-weight:600; color:#718096;'>Sentinel Standby (Mic Off)</div>")

                    # TAB 7: ROUTE WATCHDOG
                    with gr.TabItem("🗺️ Route Geofence"):
                        with gr.Column(elem_classes=["vault-panel"]):
                            gr.Markdown("### 🗺️ Live Trajectory & Safe Route Watchdog\nMonitors your GPS corridor in real-time and arms direct forward links to your emergency contacts.")
                            with gr.Row():
                                route_contact = gr.Textbox(label="Emergency Contact Number", value="+91-9876543210")
                                target_coord = gr.Textbox(label="Destination Landmark / Address", placeholder="e.g. Migsun Twinz, Metro Gate 3")
                                max_drift = gr.Slider(minimum=50, maximum=1000, value=200, step=25, label="Max Allowable Deviation (Meters)")
                            with gr.Row():
                                start_watchdog_btn = gr.Button("🧭 Start Route Watchdog", elem_classes=["primary-btn"])
                                stop_watchdog_btn = gr.Button("⏹ End Route Tracking", elem_classes=["primary-btn"])
                            gr.HTML("<div id='route_watchdog_feedback' style='padding:12px; font-weight:600; color:#718096;'>Route Watchdog Idle</div>")
                            gr.HTML("<div id='route_dispatch_container'></div>")

    # Wire User Interactions
    login_submit.click(secure_login, inputs=[login_email, login_password], outputs=[auth_panel, app_panel, auth_feedback, current_user_state]).then(render_case_history, inputs=[current_user_state], outputs=[case_history_output])
    reg_submit.click(secure_register, inputs=[reg_email, reg_password, reg_confirm], outputs=reg_feedback)
    submit_btn.click(process_evidence_and_ai, inputs=[current_user_state, cat_input, perp_input, narrative_input, file_input], outputs=[vault_status_output, case_history_output])
    gen_pdf_btn.click(generate_complaint_pdf, inputs=[current_user_state], outputs=[pdf_file_out, vault_status_output])

    # Notarization Wire
    notarize_btn.click(
        lambda email: notarize_vault_record(load_json(CASES_FILE).get(email, [{}])[-1].get("case_id", ""), email) if load_json(CASES_FILE).get(email) else "⚠️ No cases in vault.",
        inputs=[current_user_state],
        outputs=[vault_status_output]
    )

    logout_btn.click(lambda: (gr.update(visible=True), gr.update(visible=False), ""), outputs=[auth_panel, app_panel, current_user_state])

    # Stealth Camouflage Trigger & Unlock
    stealth_trigger.click(lambda: (gr.update(visible=True), gr.update(visible=False)), outputs=[disguise_panel, main_she_safe_container])
    calc_btn.click(calculator_eval, inputs=[calc_display], outputs=[calc_display, disguise_panel, main_she_safe_container])
    calc_display.submit(calculator_eval, inputs=[calc_display], outputs=[calc_display, disguise_panel, main_she_safe_container])

    # Emergency SOS with JS Hardware Polling
    sos_btn.click(fn=trigger_sos_alert, inputs=[sos_contacts, sos_note, landmark_box, live_lat_box, live_lon_box, live_acc_box], outputs=[sos_display], js=GPS_ACCURACY_LOCK_JS)

    # Fake Call Flow
    call_btn.click(start_incoming_call, inputs=[call_delay, fake_name, fake_type], outputs=[call_standby_box, phone_viewport, phone_info_html, incoming_actions_row, hangup_actions_row, current_caller_state])
    accept_call_btn.click(answer_call, inputs=[current_caller_state], outputs=[phone_info_html, incoming_actions_row, hangup_actions_row])
    decline_call_btn.click(end_call, outputs=[call_standby_box, phone_viewport, incoming_actions_row, hangup_actions_row])
    hangup_call_btn.click(end_call, outputs=[call_standby_box, phone_viewport, incoming_actions_row, hangup_actions_row])

    # Siren Audio Handlers
    siren_start_btn.click(play_siren_audio, outputs=[siren_audio_sink])
    siren_stop_btn.click(stop_siren_audio, outputs=[siren_audio_sink])

    # Live Safety Timer Handlers (Pure JS)
    timer_start_btn.click(fn=None, inputs=[timer_mins], js=TIMER_START_JS)
    timer_cancel_btn.click(fn=None, js=TIMER_CANCEL_JS)

    # Sentinel Audio Triggers
    start_sentinel_btn.click(fn=None, js=START_DISTRESS_LISTENER_JS)
    stop_sentinel_btn.click(fn=None, js=STOP_DISTRESS_LISTENER_JS)

    # Route Watchdog Triggers (Properly Indented Inside Blocks)
    start_watchdog_btn.click(fn=None, inputs=[route_contact, target_coord, max_drift], js=START_ROUTE_WATCHDOG_JS)
    stop_watchdog_btn.click(fn=None, js=STOP_ROUTE_WATCHDOG_JS)

# Launch single unified interface
gr.close_all()
app.launch(share=True, debug=False, css=custom_css)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c56dcf5a20a24bccd8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
